# Pothole Detector - YOLOv8 Training (Google Colab)

Trains the **IIT Madras Pothole Detection v2** dataset (2,722 images; 3 classes:
`crocodile crack`, `longitudinal crack`, `pothole`) on a free Colab **T4 GPU**.

## FIRST: turn on the GPU
`Runtime` -> `Change runtime type` -> **Hardware accelerator: T4 GPU** -> Save.

## Steps
1. Enable GPU (above)
2. Install Ultralytics
3. Load the dataset (upload the zip to Google Drive first)
4. Train
5. Evaluate on the held-out **test** split (honest metrics)
6. Visual sanity check
7. Download `best.pt` and drop it into the project at
   `models/pothole_detector/weights/best.pt`

## 0. Confirm the GPU is on
If this prints a table with `Tesla T4`, you're good. If it says *command not found* / no GPU, redo the **Runtime -> Change runtime type** step above.

In [ ]:
!nvidia-smi

## 1. Install Ultralytics (YOLOv8)

In [ ]:
!pip -q install ultralytics
import ultralytics
ultralytics.checks()

## 2. Get the dataset

**Upload `Pothole detection.v2i.yolov8.zip` to your Google Drive first** (drag it
into drive.google.com -> *My Drive*). Then run the two cells below to mount Drive
and unzip.

> Prefer not to upload? See the optional **Roboflow** cell at the very bottom -
> the dataset is public and can be pulled directly with an API key.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os

# EDIT this if you saved the zip somewhere other than the top of My Drive:
ZIP_PATH = '/content/drive/MyDrive/Pothole detection.v2i.yolov8.zip'

DATA_DIR = '/content/pothole'
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

print('Extracted into', DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

## 3. Write a correct `data.yaml`

The Roboflow export ships `train: ../train/images` (note the `../`), which points
*outside* the dataset folder and breaks training. We overwrite it with correct paths.

In [ ]:
data_yaml = os.path.join(DATA_DIR, 'data.yaml')
with open(data_yaml, 'w') as f:
    f.write(
        'path: /content/pothole\n'
        'train: train/images\n'
        'val: valid/images\n'
        'test: test/images\n'
        'nc: 3\n'
        "names: ['crocodile crack', 'longitudinal crack', 'pothole']\n"
    )
print(open(data_yaml).read())

## 4. Train

Defaults: `yolov8s`, 100 epochs, 640px, batch 16 -> roughly **45-90 min** on a T4.

- Want more accuracy? Set `MODEL = 'yolov8m.pt'` and/or `EPOCHS = 150` (slower).
- Training auto-stops early if it stops improving (`patience=30`).

In [ ]:
from ultralytics import YOLO

MODEL  = 'yolov8s.pt'   # 'yolov8m.pt' = more accuracy, slower
EPOCHS = 100            # 150 for a little more, if time allows

model = YOLO(MODEL)
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    device=0,            # T4 GPU
    pretrained=True,
    optimizer='auto',
    patience=30,
    cos_lr=True,
    close_mosaic=10,
    seed=42,
    deterministic=True,
    plots=True,
    project='/content/runs',
    name='pothole_detector',
    exist_ok=True,
)
print('Best weights: /content/runs/pothole_detector/weights/best.pt')

## 5. Evaluate on the held-out TEST split

This is the honest score - these images were never seen during training. Watch the
**per-class** numbers: `crocodile crack` is rare (~8% of all labels), so it will
almost certainly score lowest.

In [ ]:
metrics = model.val(split='test', data=data_yaml, imgsz=640, plots=True)

print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print(f"precision : {metrics.box.mp:.4f}")
print(f"recall    : {metrics.box.mr:.4f}")

print("\nPer-class mAP@50-95:")
for c in metrics.box.ap_class_index:
    print(f"  {metrics.names[c]:20s} {metrics.box.maps[c]:.4f}")

## 6. Visual sanity check - detect on a test image

In [ ]:
import glob
from IPython.display import Image as ColabImage

sample = sorted(glob.glob('/content/pothole/test/images/*.jpg'))[0]
model.predict(sample, conf=0.25, save=True,
              project='/content/runs', name='predict', exist_ok=True)

pred = sorted(glob.glob('/content/runs/predict/*.jpg'))[0]
ColabImage(filename=pred, width=640)

## 7. Download the trained model

Saves a copy to your Drive **and** downloads `best.pt`.

Then, in the project, put the file at:
```
models/pothole_detector/weights/best.pt
```
The FastAPI backend (`/detect-image`, `/upload-report`) will pick it up automatically.

In [ ]:
import shutil
from google.colab import files

best = '/content/runs/pothole_detector/weights/best.pt'
assert os.path.exists(best), best

shutil.copy(best, '/content/drive/MyDrive/pothole_best.pt')
print('Saved a copy to your Drive: MyDrive/pothole_best.pt')
files.download(best)

---
## (Optional) Alternative: pull the dataset from Roboflow

If you'd rather not upload the zip, the dataset is public. Get a free API key from
roboflow.com (Settings -> API), then run:

```python
!pip -q install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key='YOUR_KEY')
ds = (rf.workspace('indian-institute-of-technology-madras-xamot')
        .project('pothole-detection-huf2x')
        .version(2)
        .download('yolov8'))
data_yaml = ds.location + '/data.yaml'   # use this instead of the Drive path
```